# Check SQLite database (`excel_data.db`)

This notebook summarizes **everything persisted** by the app (`db_storage.init_db`): raw Excel structure (`files`, `sheets`, `columns`, `data`), graph edges (`relationships`), vectors (`embeddings`), and finalized S2T metadata (`source_tables`, `target_tables`, `column_mappings`, `additions`).

**Run location:** preferably from the project root (`etl-mapping-rag`) so `excel_data.db` is found next to `notebooks/` or in the cwd. Paths are resolved automatically.

**Note:** `relationships` can link endpoints in different ways depending on code path:
- `load_data_from_similarity_report` → `MAPS_TO` from **Excel `columns.column_hash`** to **`column_mappings.id`**;
- `schema_matcher.create_graph_edges_from_mapping` → `MAPS_TO` from **`columns`** to **`get_target_column_id(table, column)`** (not a DB row).

The checks below classify endpoints instead of assuming a single shape.

In [20]:
from __future__ import annotations

import sqlite3
from pathlib import Path
from IPython.display import display

import pandas as pd

# Application schema (matches db_storage.init_db)
EXPECTED_TABLES = (
    "files",
    "sheets",
    "columns",
    "data",
    "relationships",
    "embeddings",
    "source_tables",
    "target_tables",
    "column_mappings",
    "additions",
)


def resolve_db_path() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "excel_data.db",
        cwd.parent / "excel_data.db",  # if cwd is notebooks/
        cwd / ".." / "excel_data.db",
    ]
    for p in candidates:
        p = p.resolve()
        if p.is_file():
            return p
    return (cwd / "excel_data.db").resolve()


def connect_ro(db: Path) -> sqlite3.Connection:
    uri_path = db.as_posix()
    return sqlite3.connect(f"file:{uri_path}?mode=ro", uri=True)


DB_PATH = resolve_db_path()
print(f"Database: {DB_PATH}")
if not DB_PATH.is_file():
    raise FileNotFoundError(
        f"SQLite file not found: {DB_PATH}. Upload a workbook in the app or copy excel_data.db here."
    )

conn = connect_ro(DB_PATH)
conn.row_factory = sqlite3.Row

Database: /Users/nikolajabramov/PycharmProjects/etl-mapping-rag/excel_data.db


In [21]:
def list_user_tables(con: sqlite3.Connection) -> list[str]:
    q = """
    SELECT name FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """
    return [r["name"] for r in con.execute(q).fetchall()]


tables = list_user_tables(conn)
missing = [t for t in EXPECTED_TABLES if t not in tables]
extra = [t for t in tables if t not in EXPECTED_TABLES]
print("tables in sqlite_master:", tables)
print("missing vs app schema:", missing or "none")
print("extra tables (not in canonical list):", extra or "none")

tables in sqlite_master: ['additions', 'column_mappings', 'columns', 'data', 'embeddings', 'files', 'relationships', 'sheets', 'source_tables', 'target_tables']
missing vs app schema: none
extra tables (not in canonical list): none


In [22]:
rows = []
for t in sorted(set(EXPECTED_TABLES) | set(tables)):
    try:
        n = conn.execute(f'SELECT COUNT(*) AS n FROM "{t}"').fetchone()["n"]
    except sqlite3.DatabaseError:
        n = None
    rows.append({"table": t, "row_count": n})

counts_df = pd.DataFrame(rows).sort_values("table").reset_index(drop=True)
counts_df

,table,row_count
0,additions,25
1,column_mappings,1232
2,columns,96
3,data,17620
4,embeddings,0
5,files,1
6,relationships,900
7,sheets,10
8,source_tables,41
9,target_tables,98


## Referential sanity (SQLite has no FKs — we check orphans in SQL)

In [23]:
checks: list[tuple[str, str, int]] = []

checks_sql = [
    (
        "sheets -> files (missing file_hash)",
        """
        SELECT COUNT(*) AS n FROM sheets s
        LEFT JOIN files f ON f.file_hash = s.file_hash
        WHERE f.file_hash IS NULL
        """,
    ),
    (
        "columns -> sheets",
        """
        SELECT COUNT(*) AS n FROM columns c
        LEFT JOIN sheets s ON s.sheet_hash = c.sheet_hash
        WHERE s.sheet_hash IS NULL
        """,
    ),
    (
        "data.sheet_hash -> sheets",
        """
        SELECT COUNT(*) AS n FROM data d
        LEFT JOIN sheets s ON s.sheet_hash = d.sheet_hash
        WHERE s.sheet_hash IS NULL
        """,
    ),
    (
        "data.column_hash -> columns",
        """
        SELECT COUNT(*) AS n FROM data d
        LEFT JOIN columns c ON c.column_hash = d.column_hash
        WHERE c.column_hash IS NULL
        """,
    ),
    (
        "column_mappings.target_table_id -> target_tables",
        """
        SELECT COUNT(*) AS n FROM column_mappings cm
        LEFT JOIN target_tables t ON t.id = cm.target_table_id
        WHERE t.id IS NULL
        """,
    ),
    (
        "column_mappings.source_table_id -> source_tables",
        """
        SELECT COUNT(*) AS n FROM column_mappings cm
        LEFT JOIN source_tables st ON st.id = cm.source_table_id
        WHERE st.id IS NULL
        """,
    ),
]

for label, sql in checks_sql:
    n = conn.execute(sql).fetchone()["n"]
    checks.append((label, "orphan_rows", int(n)))

checks_df = pd.DataFrame(checks, columns=["check", "kind", "count"])
checks_df

,check,kind,count
0,sheets -> files (missing file_hash),orphan_rows,0
1,columns -> sheets,orphan_rows,0
2,data.sheet_hash -> sheets,orphan_rows,0
3,data.column_hash -> columns,orphan_rows,2
4,column_mappings.target_table_id -> target_tables,orphan_rows,0
5,column_mappings.source_table_id -> source_tables,orphan_rows,0


In [24]:
# Precompute id sets for relationship endpoint typing
col_hashes = {r["column_hash"] for r in conn.execute("SELECT column_hash FROM columns")}
mapping_ids = {r["id"] for r in conn.execute("SELECT id FROM column_mappings")}

rel_summary = pd.read_sql_query(
    """
    SELECT relation_type AS type, COUNT(*) AS cnt
    FROM relationships GROUP BY relation_type ORDER BY cnt DESC
    """,
    conn,
)
rel_summary

,type,cnt
0,MAPS_TO,874
1,DERIVED_FROM,26


In [25]:
rows_eps = []
for row in conn.execute(
    "SELECT from_id, to_id, relation_type FROM relationships"
).fetchall():
    fid, tid = row["from_id"], row["to_id"]
    rows_eps.append(
        {
            "relation_type": row["relation_type"],
            "from_in_columns": fid in col_hashes,
            "to_in_columns": tid in col_hashes,
            "from_in_mappings": fid in mapping_ids,
            "to_in_mappings": tid in mapping_ids,
        }
    )

eps_df = pd.DataFrame(rows_eps)
if not eps_df.empty:
    frac_cols = [
        "from_in_columns",
        "to_in_columns",
        "from_in_mappings",
        "to_in_mappings",
    ]
    display(
        eps_df.groupby("relation_type")[frac_cols]
        .mean()
        .round(3)
        .rename(columns=lambda c: "frac_" + c)
    )
    # Rows where neither endpoint is a known column / mapping row (might be synthetic target column id)
    weird = eps_df[
        ~(
            (eps_df["from_in_columns"] | eps_df["from_in_mappings"])
            & (eps_df["to_in_columns"] | eps_df["to_in_mappings"])
        )
    ]
    print(
        "relationships with unresolved endpoints (empty if all edges touch DB columns/mappings):",
        len(weird),
    )
else:
    print("No relationships")

,frac_from_in_columns,frac_to_in_columns,frac_from_in_mappings,frac_to_in_mappings
relation_type,,,,
DERIVED_FROM,0.0,1.0,0.0,0.00
MAPS_TO,1.0,0.0,0.0,0.97


relationships with unresolved endpoints (empty if all edges touch DB columns/mappings): 52


## Source → mapping → target

Denormalized view of finalized S2T rows: **`source_tables`** and **`target_tables`** joined through **`column_mappings`**.

The second query adds Excel-side context when **`MAPS_TO`** links a `columns.column_hash` to a `column_mappings.id` (from `load_data_from_similarity_report`).

In [26]:
src_tgt_sql = """
SELECT
    st.name AS source_table,
    st.system_code AS source_system_code,
    st.description AS source_description,
    cm.source_column,
    tt.name AS target_table,
    cm.target_column,
    tt.description AS target_description,
    cm.transformation_rule,
    cm.data_type,
    cm.is_primary_key,
    cm.id AS mapping_id,
    cm.source_table_id,
    cm.target_table_id
FROM column_mappings cm
INNER JOIN source_tables st ON st.id = cm.source_table_id
INNER JOIN target_tables tt ON tt.id = cm.target_table_id
ORDER BY
    COALESCE(tt.name, ''),
    COALESCE(cm.target_column, ''),
    COALESCE(st.name, ''),
    COALESCE(cm.source_column, '')
"""

mapping_join_df = pd.read_sql_query(src_tgt_sql, conn)
print("rows (inner join source × mapping × target):", len(mapping_join_df))
display(mapping_join_df.head(50))

orph_cols = pd.read_sql_query(
    """
    SELECT COUNT(*) AS n FROM column_mappings cm
    LEFT JOIN source_tables st ON st.id = cm.source_table_id
    LEFT JOIN target_tables tt ON tt.id = cm.target_table_id
    WHERE st.id IS NULL OR tt.id IS NULL
    """,
    conn,
).iloc[0]["n"]
print("column_mappings rows missing source_tables or target_tables (should be 0):", int(orph_cols))

# Enrich with Excel columns when lineage MAPS_TO(column_hash -> mapping_id) exists
with_excel_sql = """
SELECT
    sh.sheet_name AS excel_sheet,
    c.column_index,
    c.column_name_flat AS excel_column_flat,
    st.name AS source_table,
    cm.source_column,
    tt.name AS target_table,
    cm.target_column,
    cm.id AS mapping_id,
    cm.transformation_rule
FROM column_mappings cm
INNER JOIN source_tables st ON st.id = cm.source_table_id
INNER JOIN target_tables tt ON tt.id = cm.target_table_id
LEFT JOIN relationships r ON r.to_id = cm.id AND r.relation_type = 'MAPS_TO'
LEFT JOIN columns c ON c.column_hash = r.from_id
LEFT JOIN sheets sh ON sh.sheet_hash = c.sheet_hash
ORDER BY
    excel_sheet,
    excel_column_flat,
    target_table,
    cm.target_column
"""

mapping_join_excel_df = pd.read_sql_query(with_excel_sql, conn)
_with_lineage = (
    mapping_join_excel_df["excel_column_flat"].notna()
    & (mapping_join_excel_df["excel_column_flat"] != "")
).sum()
print("enriched rows with Excel column lineage:", int(_with_lineage), "of", len(mapping_join_excel_df))
display(mapping_join_excel_df.head(50))

rows (inner join source × mapping × target): 1232


,source_table,source_system_code,source_description,source_column,target_table,target_column,target_description,transformation_rule,data_type,is_primary_key,mapping_id,source_table_id,target_table_id
0,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,bank_correspondentAccount_escrow_accountDetails,NaN,accountDetails.escrowAccount.bank.corresponden...,string,0,af7ee08b85809be93d01d44b7cc2b9ab,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
1,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,bankunitcode_beneficiary,NaN,beneficiary.bankUnitCode,string,0,44b49074708ddc29be708128fdefa9e9,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
2,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,closeDate_agreement,NaN,agreement.closeDate,date,0,85bd8d4c9187b8a89acf99f0dec86756,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
3,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,completeEstimateDate_agreementObject,NaN,agreementObject.completeEstimateDate,date,0,da04e5c7d8032ad454f8b8d30dd516c7,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
4,escrow_legal214_agreement_actual,None,None,deleted,escrow_legal214_agreement_actual_dto,deleted,NaN,NaN,boolean,0,10f6ce1427a877ac72132f8653d337a0,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
5,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,devLoanAgrKulId_agreement,NaN,agreement.developerLoanAgreementKulId,string,0,051e44c98fde56def0d7a11a14b2825e,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
6,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,devLoanAgr_EksId_agreement,NaN,agreement.developerLoanAgreementEksId,string,0,e9268e7f80f281229b56fc661e66d214,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
7,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,developerLoanAgrDate_cred_agreement,NaN,agreement.developerLoanAgreementDate,date,0,38b26fc317dc2744aac6467d3430ef81,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
8,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,developerLoanAgrNumber_cred_agreement,NaN,agreement.developerLoanAgreementNumber,string,0,d5d700e524a27bafe913f3d977efd732,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd
9,escrow_legal214_agreement_actual,None,None,dto,escrow_legal214_agreement_actual_dto,eksAccountId_accountDetails,NaN,accountDetails.eksAccountId,string,0,114ea5c02492363cfe142ee729267d3f,0d24e60bed20cc90a26e9a060676644d,e2e08633e679d4a8deec906a3e08f2cd


column_mappings rows missing source_tables or target_tables (should be 0): 0
enriched rows with Excel column lineage: 848 of 1232


,excel_sheet,column_index,excel_column_flat,source_table,source_column,target_table,target_column,mapping_id,transformation_rule
0,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,agreementReference,9204734d83f71ec9a9c72d78a26bfb0b,agreementReference
1,NaN,NaN,NaN,percent_payment,deleted,percent_payment_dto,deleted,0f34132456b58f8b6bdf9e796032d74f,NaN
2,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,eventStatus,4b466f7dc83bcab301c8766fc8ac90e1,eventStatus
3,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,factoryId,b7abbd98b103e5d40074713a08eff055,factoryId
4,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,feeEventType_code,be97c3958401c6c3256bab1332bfadac,feeEventType.code
5,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,feeEventType_description,5fcdb9b786e5101946abbf61cff69877,feeEventType.description
6,NaN,NaN,NaN,percent_payment,id,percent_payment_dto,id,0d73ebd55abeb836d2895d522bff23b8,NaN
7,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,scheduleEntry_dateFrom,6e42f45ef4d074cd71a0885604670b61,scheduleEntry.dateFrom
8,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,scheduleEntry_dateTill,c6bdf6a83ea94b907336e6fa93f79228,scheduleEntry.dateTill
9,NaN,NaN,NaN,percent_payment,dto,percent_payment_dto,scheduleEntry_scheduleAmount_currency,6e575c2e826827a244ca7fcf20402dd7,scheduleEntry.scheduleAmount.currency


## Embeddings (`entity_type` mix + orphan `entity_id` vs `columns`)

In [27]:
emb_by_type = pd.read_sql_query(
    "SELECT entity_type, COUNT(*) AS n FROM embeddings GROUP BY entity_type ORDER BY n DESC",
    conn,
)
display(emb_by_type)

_emb_tot = pd.read_sql_query("SELECT COUNT(*) AS n FROM embeddings", conn).iloc[0]["n"]
_emb_hit = pd.read_sql_query(
    """
    SELECT COUNT(*) AS n FROM embeddings e
    WHERE EXISTS (SELECT 1 FROM columns c WHERE c.column_hash = e.entity_id)
    """,
    conn,
).iloc[0]["n"]
print(
    "embeddings whose entity_id matches a columns.column_hash:",
    f"{_emb_hit}/{_emb_tot}" if _emb_tot else "0/0",
)

emb_orphan_typed_column = pd.read_sql_query(
    """
    SELECT COUNT(*) AS n FROM embeddings e
    WHERE LOWER(e.entity_type) = 'column'
      AND NOT EXISTS (SELECT 1 FROM columns c WHERE c.column_hash = e.entity_id)
    """,
    conn,
).iloc[0]["n"]
print(
    "embeddings typed 'column' with no matching columns.column_hash:",
    int(emb_orphan_typed_column),
)

,entity_type,n


embeddings whose entity_id matches a columns.column_hash: 0/0
embeddings typed 'column' with no matching columns.column_hash: 0


## Per-file rollup (excel uploads)

In [28]:
files_rollup = pd.read_sql_query(
    """
    SELECT
        f.file_hash,
        f.filename,
        f.model_used,
        f.upload_time,
        CASE WHEN f.summary IS NOT NULL AND TRIM(f.summary) != '' THEN 1 ELSE 0 END AS has_summary,
        CASE WHEN f.result_json IS NOT NULL AND TRIM(f.result_json) != '' THEN 1 ELSE 0 END AS has_result_json,
        (SELECT COUNT(*) FROM sheets sh WHERE sh.file_hash = f.file_hash AND IFNULL(sh.skipped, 0) = 0)
            AS sheets_loaded,
        (SELECT COUNT(*) FROM sheets sh WHERE sh.file_hash = f.file_hash AND IFNULL(sh.skipped, 0) != 0)
            AS sheets_skipped,
        (SELECT COUNT(DISTINCT c.column_hash)
         FROM sheets sh JOIN columns c ON c.sheet_hash = sh.sheet_hash
         WHERE sh.file_hash = f.file_hash AND IFNULL(sh.skipped, 0) = 0
        ) AS column_count,
        (SELECT COUNT(*)
         FROM data d
         JOIN sheets sh ON sh.sheet_hash = d.sheet_hash
         WHERE sh.file_hash = f.file_hash AND IFNULL(sh.skipped, 0) = 0
        ) AS cell_count
    FROM files f ORDER BY COALESCE(f.upload_time, '') DESC
    """,
    conn,
)
files_rollup

,file_hash,filename,model_used,upload_time,has_summary,has_result_json,sheets_loaded,sheets_skipped,column_count,cell_count
0,1c5bf176656a2076954c34f6777b972f,S2T_308_000046_v0.39.xlsx,GigaChat-Pro,2026-05-08T06:42:17.549431,1,1,9,1,96,17620


## Samples (first 8 rows each)

In [29]:
for t in EXPECTED_TABLES:
    try:
        sample = pd.read_sql_query(f'SELECT * FROM "{t}" LIMIT 8', conn)
    except Exception as exc:  # noqa: BLE001
        print(t, "->", exc)
        continue
    print("\n===", t, "===")
    display(sample)


=== files ===


,file_hash,filename,model_used,upload_time,summary,result_json
0,1c5bf176656a2076954c34f6777b972f,S2T_308_000046_v0.39.xlsx,GigaChat-Pro,2026-05-08T06:42:17.549431,Файл представляет собой детальную документацию...,"{""filename"": ""S2T_308_000046_v0.39.xlsx"", ""mod..."



=== sheets ===


,sheet_hash,file_hash,sheet_name,header_start_row,header_rows_count,nested_structure,skipped,skip_reason
0,fa828d25e9d8cda92801e4434ec7fc04,1c5bf176656a2076954c34f6777b972f,Список изменений,0,1,0,0,None
1,b267fb38b56b592db97308cc85fdedf7,1c5bf176656a2076954c34f6777b972f,S2T,1,1,0,0,None
2,9c193ba4cdbfd478947ca7fa2e4e03aa,1c5bf176656a2076954c34f6777b972f,Source columns,1,1,0,0,None
3,0c0d0e767120f823e343688e293b72f5,1c5bf176656a2076954c34f6777b972f,Target tables,1,1,0,0,None
4,74571462e568c6d6a3bff39e62d604e0,1c5bf176656a2076954c34f6777b972f,Target columns,1,1,0,0,None
5,ca72e10ff814d62c429d163e2a79085d,1c5bf176656a2076954c34f6777b972f,Метаданные для hadoop,0,1,0,0,None
6,4edffa608c3c29ab8b5e5957b8ba08c0,1c5bf176656a2076954c34f6777b972f,hadoop ->A,0,1,0,0,None
7,abd42ca3629e52d0c4e113151356df7b,1c5bf176656a2076954c34f6777b972f,А-слой->В-слой->S-слой,0,2,1,0,None



=== columns ===


...

,id,name,description
0,fc59e0ebae51e1da89c9fb681d56e789,t_agr_escrow,None
1,930ceabc2e8d15595e4fa858459a5c46,t_agr_cred,None
2,b6961befb46f71c52bdc45a48b7c4779,t_registry,None
3,c7747290c3743a72c849213b46e0561b,t_escrow_scheme,None
4,7eee79f5ce2ef1b69ccabd3fac3fcd0a,t_agr_escrow_agr_cred,None
5,db4e667c61bb7176619f8fd931cc0492,t_agr_escrow_cust,None
6,6ef3c773274277113f564a32873ce068,t_cust,None
7,6f9af27785c7638059a2ac323fec4fc3,t_int_org,None



=== column_mappings ===


,id,target_table_id,target_column,source_table_id,source_column,transformation_rule,data_type,is_primary_key
0,57b8ed4ac45da528f3235366d87a1834,fc59e0ebae51e1da89c9fb681d56e789,agr_escrow_id,2b4ea93267ae9196aff4eb241f3be1f0,Идентификатор договора эскроу,escrowId_agreement_uid,UUID,1
1,d97a0665f55b96465cecbac1ef60e9ea,fc59e0ebae51e1da89c9fb681d56e789,open_dt,2b4ea93267ae9196aff4eb241f3be1f0,Дата открытия договора эскроу,openDate_agreement,DATE,0
2,3825bf1df79d7fc941221863b9792157,fc59e0ebae51e1da89c9fb681d56e789,signed_dt,2b4ea93267ae9196aff4eb241f3be1f0,Дата открытия договора эскроу,openDate_agreement,DATE,0
3,ae32f6764c69c45762ec28ae898e5093,fc59e0ebae51e1da89c9fb681d56e789,close_dt,2b4ea93267ae9196aff4eb241f3be1f0,Дата закрытия договора эскроу,closeDate_agreement,DATE,0
4,32f752a6d08753e0c3755f90dac29d78,fc59e0ebae51e1da89c9fb681d56e789,agr_escrow_num,2b4ea93267ae9196aff4eb241f3be1f0,Номер договора,reference_agreement,TEXT,0
5,8b9b69e0881dfb11b99bc550567bc1a2,fc59e0ebae51e1da89c9fb681d56e789,host_agr_escrow_id,2b4ea93267ae9196aff4eb241f3be1f0,Идентификатор договора эскроу,escrowId_agreement,TEXT,0
6,7a32f922f2751603d3d65663ba5493e5,fc59e0ebae51e1da89c9fb681d56e789,permission_explotation_plan_dt,2b4ea93267ae9196aff4eb241f3be1f0,Плановая дата ввода ОС в эксплуатацию,completeEstimateDate_agreementObject,DATE,0
7,d691250c190401cf6634baa6d645c0f3,fc59e0ebae51e1da89c9fb681d56e789,depositing_object_dt,2b4ea93267ae9196aff4eb241f3be1f0,Срок депонирования объекта,estateObject_ExpireDate_agreementObject,DATE,0



=== additions ===


,id,table_name,table_description,source_tables_name,sql,description
0,8f6e0bc62f923ffebb0647c5248aa0fb,None,Снимок-снимок,a_000046_escrow_legal214_agreement_actual_dto_...,None,a_000046_escrow_legal214_agreement_actual_dto
1,d1af9b3547314ab2684276bff2bc33b9,None,Снимок-снимок,a_000046_escrow_legalgk_agreement_actual_dto_s...,None,a_000046_escrow_legalgk_agreement_actual_dto
2,8c6d40937872a281b6aa2aeea1bfe528,None,Снимок-снимок,a_000046_escrow_legalgk_beneficiariesindividualс,None,a_000046_escrow_legalgk_beneficiariesindividual
3,53517ed58d5ff4530c53611e2ac12f86,None,Снимок-снимок,a_000046_escrow_legalgk_beneficiarieslegal_snp...,None,a_000046_escrow_legalgk_beneficiarieslegal
4,37c6ebb39b881be4815fb6b8c5a8a114,None,Снимок-снимок,a_000046_escrow_legalgk_tariffs_snp_ext,None,a_000046_escrow_legalgk_tariffs
5,23ed8fda03ef6856d01765ab98f2d1f4,None,Снимок-снимок,a_000046_payments_2_0_posting_dto_snp_ext,None,a_000046_payments_2_0_posting_dto
6,f02181eaa956a7e30e95d9c45e719828,None,Снимок-снимок,a_000046_general_agreement_actual_dto_snp_ext,None,a_000046_general_agreement_actual_dto
7,8d60b6805a741cfbc8ef672e714895d9,None,Снимок-снимок,a_000046_general_agreement_actual_limitschedul...,None,a_000046_general_agreement_actual_limitschedule


In [30]:
conn.close()
print("Read-only connection closed.")

Read-only connection closed.
